<a href="https://colab.research.google.com/github/yoshiagent/automaticControl-colab/blob/main/notebooks/SteadyState_Error.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="在 Colab 開啟"/></a>

# 穩態誤差：系統型別與 P／I 控制

對照網頁：[自動控制｜穩態誤差](https://yoshiagent.github.io/automaticControl/ESS.html)

原課程用 5 個 Simulink 模型（`.mdl`）示範穩態誤差。這本筆記本把每個模型的方塊與參數（直接從 `.mdl` 檔讀出，
沒寫的參數用 Simulink 預設值：Gain＝1、Step 在 $t=1$ 由 0 跳到 1、Ramp 斜率 1 從 $t=0$ 開始）改寫成 Python，
再和網頁上的公式 $e_{ss}=\lim_{s\to0}\dfrac{s}{1+G(s)}R(s)$ 比對。所有模型都是單位負回授，Scope 用 Mux 同時畫出 $r(t)$ 與 $y(t)$。

| 節 | 原檔 | 內容 |
|---|---|---|
| 1 | `Ess_NoSpring.mdl` | 有／沒有彈簧：Type 0 與 Type 1 對步階輸入 |
| 2 | `Ess_inf.mdl` | Type 0 對斜坡輸入：誤差無限大 |
| 3 | `Ess_Type1_2.mdl`、`Ess_Type2Step.mdl` | Type 1、Type 2 對斜坡；Type 2 對步階 |
| 4 | `Ess_Eliminate.mdl` | 提高 $K$ 只能減小、加積分器才能消除步階誤差 |

只用 Colab 內建的 numpy、scipy、plotly。圖上的文字用英文。

In [1]:
import numpy as np
from scipy import signal
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def step(t, t0=1.0, after=1.0, before=0.0):
    """Simulink 的 Step 方塊（預設 Step time = 1、Initial value = 0、Final value = 1）"""
    return after if t >= t0 else before


def ramp(t, slope=1.0, start=0.0):
    """Simulink 的 Ramp 方塊（預設 Slope = 1、Start time = 0）"""
    return slope * (t - start) if t >= start else 0.0


class TF:
    """Simulink 的 Transfer Fcn 方塊：num、den 是 s 多項式係數（高次在前，同 MATLAB）。
    內部轉成狀態空間 x' = Ax + Bu、y = Cx + Du；n 是狀態數（＝分母階數）。"""
    def __init__(self, num, den):
        A, B, C, D = signal.tf2ss(num, den)
        self.A, self.B, self.C, self.D = A, B[:, 0], C[0], D[0, 0]
        self.n = len(A)
        self.num, self.den = np.atleast_1d(num).astype(float), np.atleast_1d(den).astype(float)

    def dx(self, x, u):
        return self.A @ x + self.B * u

    def y(self, x, u=0.0):
        return self.C @ x + self.D * u


def simulate(rhs, x0, T, breaks=(), dt=0.01, max_step=0.05):
    """仿 Simulink 從 t = 0 跑到 T。breaks：輸入跳變的時刻（例如 Step），在這些點分段積分，
    跳變前那一段一律用跳變前的輸入值。回傳等間隔 dt 的 t 與狀態陣列 X（X[i] 是第 i 個狀態的時間序列）。"""
    n = round(T / dt)
    t = np.linspace(0, T, n + 1)
    cuts = [0.0] + sorted(float(b) for b in breaks if 0 < b < T) + [float(T)]
    x = np.array(x0, dtype=float)
    X = np.empty((len(x), n + 1))
    for a, b in zip(cuts[:-1], cuts[1:]):
        tb = np.nextafter(b, a)
        sol = solve_ivp(lambda s, z: rhs(min(s, tb), z), (a, b), x, method="RK45",
                        rtol=1e-9, atol=1e-12, max_step=max_step, dense_output=True)
        i0, i1 = round(a / dt), round(b / dt)
        idx = slice(i0, i1 + 1 if b == T else i1)
        X[:, idx] = sol.sol(t[idx])
        x = sol.y[:, -1]
    return t, X


def scope(t, panels, title, h=None, npts=600):
    """像 Simulink 的 Scope：panels = {子圖標題: [(曲線名稱, 數列), …]}。點數太多時等距抽樣到約 npts 點（只影響畫圖）。"""
    k = max(1, int(np.ceil(len(t) / npts)))
    fig = make_subplots(rows=len(panels), cols=1, shared_xaxes=True, subplot_titles=list(panels), vertical_spacing=0.1)
    for i, traces in enumerate(panels.values(), 1):
        for name, y in traces:
            fig.add_scatter(x=np.round(t[::k], 3).tolist(), y=np.round(np.asarray(y)[::k], 4).tolist(),
                            name=name, mode="lines", row=i, col=1)
    fig.update_xaxes(title_text="t (s)", row=len(panels), col=1)
    fig.update_layout(title=title, height=h or 260 + 220 * len(panels), template="plotly_white",
                      margin=dict(l=60, r=20, t=70, b=50))
    fig.show()


def unity_loop(G, r, T, breaks=(1,), K=1.0):
    """單位負回授：Sum "|+-"（e = r − y）→ Gain K → G(s)。回傳 t、r(t)、y(t)、e(t)。"""
    t, X = simulate(lambda t, x: G.dx(x, K * (r(t) - G.y(x))), np.zeros(G.n), T, breaks=breaks)
    rv = np.array([r(s) for s in t])
    y = G.C @ X
    return t, rv, y, rv - y


def error_constants(G):
    """Kp = lim G(s)、Kv = lim sG(s)、Ka = lim s²G(s)（s → 0）：數分母有幾個 s 因子（型別 N）"""
    num, den = np.trim_zeros(G.num, "f"), np.trim_zeros(G.den, "f")
    N = len(den) - len(np.trim_zeros(den, "b"))                       # 分母常數項起連續為 0 的個數
    lead = num[-1] / den[-1 - N]                                      # 去掉 s^N 後的 s → 0 極限
    K = [np.inf] * 3
    for i in range(3):
        K[i] = lead if i == N else (np.inf if i < N else 0.0)
    return N, K

## 1. 彈簧的影響 `Ess_NoSpring.mdl`

受控體 $m\ddot x+c\dot x+kx=f$，$m=c=1$。有彈簧（$k=1$）：$G(s)=\dfrac{1}{s^2+s+1}$，Type 0，
$K_p=1$ ⇒ 步階誤差 $e_{ss}=\dfrac{1}{1+K_p}=0.5$。沒有彈簧（$k=0$）：$G(s)=\dfrac{1}{s^2+s}=\dfrac{1}{s(s+1)}$，Type 1 ⇒ $e_{ss}=0$。
兩個迴路都是 Step 輸入、模擬 50 秒。

In [2]:
panels = {}
for name, G in [("with spring 1/(s^2+s+1)", TF([1], [1, 1, 1])), ("no spring 1/(s^2+s)", TF([1], [1, 1, 0]))]:
    N, (Kp, Kv, Ka) = error_constants(G)
    t, rv, y, e = unity_loop(G, step, 50)
    print(f"Ess_NoSpring {name:24s}: type {N}, Kp = {Kp}, e(50) = {np.round(e[-1], 4) + 0.0:.4f} (theory {1 / (1 + Kp):.4f})")
    panels[name] = [("r", rv), ("y", y)]
scope(t, panels, "Ess_NoSpring.mdl: step input")

Ess_NoSpring with spring 1/(s^2+s+1) : type 0, Kp = 1.0, e(50) = 0.5000 (theory 0.5000)
Ess_NoSpring no spring 1/(s^2+s)     : type 1, Kp = inf, e(50) = 0.0000 (theory 0.0000)


沒有彈簧時，只要還有一點誤差，控制力就會推動質量（沒有彈簧力抵抗），直到誤差為 0；
有彈簧時，彈簧力要靠誤差產生的控制力來平衡，所以必須留下穩態誤差。

## 2. Type 0 系統的斜坡輸入 `Ess_inf.mdl`

$G(s)=\dfrac{K}{s^2+s+1}$（$K=1$），輸入 Ramp $r=t$，模擬 20 秒。Type 0 的 $K_v=0$ ⇒ $e_{ss}=\dfrac{1}{K_v}=\infty$：
輸出跟不上斜坡，誤差一直變大。部分分式可算出 $t$ 大時 $e(t)\approx\dfrac t2+\dfrac14$。

In [3]:
G = TF([1], [1, 1, 1])
t, rv, y, e = unity_loop(G, ramp, 20, breaks=())
print(f"Ess_inf: e(10) = {e[t <= 10][-1]:.4f}, e(20) = {e[-1]:.4f}; t/2 + 1/4 at t = 20 is {20 / 2 + 0.25:.4f}")
scope(t, {"Mux: ramp r and output y": [("r = t", rv), ("y", y)], "error e = r - y": [("e", e)]}, "Ess_inf.mdl: type 0, ramp input")

Ess_inf: e(10) = 5.2498, e(20) = 10.2500; t/2 + 1/4 at t = 20 is 10.2500


## 3. Type 1、Type 2 系統 `Ess_Type1_2.mdl`、`Ess_Type2Step.mdl`

* Type 1：$G(s)=\dfrac{1}{s^2+s}$，$K_v=\lim sG=1$ ⇒ 斜坡誤差 $e_{ss}=\dfrac{1}{K_v}=1$。
* Type 2：$G(s)=\dfrac{2s+1}{s^3+s^2}=\dfrac{2s+1}{s^2(s+1)}$，$K_a=\lim s^2G=1$ ⇒ 斜坡誤差 $e_{ss}=0$（拋物線輸入才有 $\dfrac{1}{K_a}$）；步階誤差也是 0。

閉迴路特性方程式 Type 2 為 $s^3+s^2+2s+1=0$（$1\times2>1$，穩定）。三個迴路都模擬 50 秒。

In [4]:
G1, G2 = TF([1], [1, 1, 0]), TF([2, 1], [1, 1, 0, 0])
panels = {}
for name, G, r, br in [("type 1, ramp", G1, ramp, ()), ("type 2, ramp", G2, ramp, ()), ("type 2, step (Ess_Type2Step)", G2, step, (1,))]:
    N, (Kp, Kv, Ka) = error_constants(G)
    t, rv, y, e = unity_loop(G, r, 50, breaks=br)
    print(f"{name:28s}: type {N}, Kp = {Kp}, Kv = {Kv}, Ka = {Ka}, e(50) = {np.round(e[-1], 4) + 0.0:.4f}")
    panels[name] = [("r", rv), ("y", y)]
print("type 2 closed-loop poles:", 0 + np.round(np.roots([1, 1, 2, 1]), 4))
scope(t, panels, "Ess_Type1_2.mdl and Ess_Type2Step.mdl")

type 1, ramp                : type 1, Kp = inf, Kv = 1.0, Ka = 0.0, e(50) = 1.0000
type 2, ramp                : type 2, Kp = inf, Kv = inf, Ka = 1.0, e(50) = 0.0000


type 2, step (Ess_Type2Step): type 2, Kp = inf, Kv = inf, Ka = 1.0, e(50) = 0.0000
type 2 closed-loop poles: [-0.2151+1.3071j -0.2151-1.3071j -0.5698+0.j    ]


和網頁上 Dorf & Bishop 的表一致：型別每多一個積分器，就多一種輸入的穩態誤差變成 0。

| 型別 | 步階 | 斜坡 | 拋物線 |
|---|---|---|---|
| 0 | $\frac{1}{1+K_p}$ | $\infty$ | $\infty$ |
| 1 | 0 | $\frac{1}{K_v}$ | $\infty$ |
| 2 | 0 | 0 | $\frac{1}{K_a}$ |

## 4. 消除 Type 0 系統的步階誤差 `Ess_Eliminate.mdl`

模型上下兩個迴路，受控體都是 $G(s)=\dfrac{1}{s^2+s+1}$，模擬 50 秒：

* 上：P 控制 $K=3$ ⇒ $K_p=K$，$e_{ss}=\dfrac{1}{1+K}=0.25$。網頁說 Try $K=1,3,\dots$：$K$ 越大誤差越小，但永遠不是 0。
* 下：$K_I=0.5$ → 積分器 → $G(s)$，開迴路 $\dfrac{K_I}{s(s^2+s+1)}$ 變成 Type 1 ⇒ $e_{ss}=0$。
  網頁說 Try $K_I=1$、$K_I=2$：特性方程式 $s^3+s^2+s+K_I=0$，Routh 條件 $1\times1>K_I$，$K_I=1$ 臨界穩定、$K_I=2$ 不穩定。

In [5]:
G = TF([1], [1, 1, 1])
panels = {"P control (K = 1, 3, 10)": []}
for K in (1, 3, 10):
    t, rv, y, e = unity_loop(G, step, 50, K=K)
    print(f"Ess_Eliminate P control K = {K:2d}: e(50) = {np.round(e[-1], 4) + 0.0:.4f} (theory 1/(1+K) = {1 / (1 + K):.4f})")
    panels["P control (K = 1, 3, 10)"].append((f"K = {K}", y))


def i_loop(KI, T=50):
    def rhs(t, s):
        x, z = s[:G.n], s[G.n]
        e = step(t) - G.y(x)
        return np.r_[G.dx(x, z), KI * e]                              # Sum1 → KI → Integrator → G(s)
    t, X = simulate(rhs, np.zeros(G.n + 1), T, breaks=(1,))
    return t, G.C @ X[:G.n]


panels["I control (KI = 0.5, 1, 2)"] = []
for KI in (0.5, 1, 2):
    t, y = i_loop(KI)
    poles = np.roots([1, 1, 1, KI])
    print(f"Ess_Eliminate I control KI = {KI}: max Re(pole) = {np.round(poles.real.max(), 4) + 0.0:+.4f}, y(50) = {y[-1]:.4f}")
    panels["I control (KI = 0.5, 1, 2)"].append((f"KI = {KI}", y))
scope(t, panels, "Ess_Eliminate.mdl: raising K vs adding an integrator")

Ess_Eliminate P control K =  1: e(50) = 0.5000 (theory 1/(1+K) = 0.5000)


Ess_Eliminate P control K =  3: e(50) = 0.2500 (theory 1/(1+K) = 0.2500)


Ess_Eliminate P control K = 10: e(50) = 0.0909 (theory 1/(1+K) = 0.0909)


Ess_Eliminate I control KI = 0.5: max Re(pole) = -0.1761, y(50) = 1.0001


Ess_Eliminate I control KI = 1: max Re(pole) = +0.0000, y(50) = 1.3266
Ess_Eliminate I control KI = 2: max Re(pole) = +0.1766, y(50) = 1184.5276


$K_I=0.5$ 時穩態誤差為 0；$K_I=1$ 時極點 $\pm j$ 在虛軸上，持續振盪；$K_I=2$ 振盪發散。
網頁最後的問題「How to choose $K_I$?」答案是根軌跡：畫出 $1+K_I\dfrac{1}{s(s^2+s+1)}=0$ 的根隨 $K_I$ 的變化，
下面標出三個 $K_I$ 的閉迴路極點。

In [6]:
KIs = np.linspace(0, 3, 301)
roots = np.array([np.roots([1, 1, 1, k]) for k in KIs])
fig = go.Figure()
fig.add_scatter(x=np.round(roots.real.ravel(), 4).tolist(), y=np.round(roots.imag.ravel(), 4).tolist(), mode="markers",
                marker=dict(size=3, color="#1f77b4"), name="root locus, KI = 0 ~ 3")
for KI, c in [(0.5, "#2ca02c"), (1, "#ff7f0e"), (2, "#d62728")]:
    p = np.roots([1, 1, 1, KI])
    fig.add_scatter(x=p.real, y=p.imag, mode="markers", marker=dict(size=10, color=c, symbol="x"), name=f"KI = {KI}")
fig.add_vline(x=0, line=dict(color="gray", dash="dot"))
fig.update_layout(title="Root locus of KI / (s(s^2+s+1))", xaxis_title="Re", yaxis_title="Im", height=480,
                  template="plotly_white", yaxis=dict(scaleanchor="x"))
fig.show()